# Scales and Categorical Data

This notebook demonstrates how to work with scales, in particular ordinal/categorical data, in pandas. This provides efficiency and correct logical sorting that alphabetical characters wouldn't usually have.

In [2]:
import pandas as pd
import numpy as np

### Categorical Data

Let's create a DataFrame corresponding to letter grades for students. By default, pandas treats these as plain objects/strings.

In [3]:
df=pd.DataFrame(['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D+', 'D'],
                index=['excellent', 'excellent', 'excellent', 'good', 'good', 'good', 
                       'ok', 'ok', 'ok', 'poor', 'poor'],
               columns=["Grades"])
df

,Grades
excellent,A+
excellent,A
excellent,A-
good,B+
good,B
good,B-
ok,C+
ok,C
ok,C-
poor,D+


In [4]:
df.dtypes

Grades    object
dtype: object

We can explicitly convert this column into a `category` type using `.astype()`. Under the hood, pandas maps categories to efficient integer sequences.

In [5]:
df["Grades"].astype("category").head()

excellent    A+
excellent     A
excellent    A-
good         B+
good          B
Name: Grades, dtype: category
Categories (11, object): ['A', 'A+', 'A-', 'B', ..., 'C+', 'C-', 'D', 'D+']

However, grades have an implicit order (A > B > C). We can define a `CategoricalDtype` explicitly to enforce this logical ordering, making boolean comparisons possible.

In [6]:
my_categories=pd.CategoricalDtype(categories=['D', 'D+', 'C-', 'C', 'C+', 'B-', 'B', 'B+', 'A-', 'A', 'A+'], ordered=True)

grades=df["Grades"].astype(my_categories)
grades.head()

excellent    A+
excellent     A
excellent    A-
good         B+
good          B
Name: Grades, dtype: category
Categories (11, object): ['D' < 'D+' < 'C-' < 'C' ... 'B+' < 'A-' < 'A' < 'A+']

If we compare the original string column sequentially, we get lexicographically (alphabetically) correct but logically wrong results (e.g., 'D' is greater than 'C' alphabetically).

In [7]:
df[df["Grades"]>"C"]

,Grades
ok,C+
ok,C-
poor,D+
poor,D


With our explicitly ordered categorical series, asking for grades `> 'C'` returns logically correct bounds, fetching C+, B, A, etc.

In [8]:
grades[grades>"C"]

excellent    A+
excellent     A
excellent    A-
good         B+
good          B
good         B-
ok           C+
Name: Grades, dtype: category
Categories (11, object): ['D' < 'D+' < 'C-' < 'C' ... 'B+' < 'A-' < 'A' < 'A+']

### Discretizing Data with pd.cut

We often want to reduce a continuous variable into categorical bins. Let's load the census data and find the average population of each state.

In [9]:
df = pd.read_csv("datasets/census.csv")

df = df[df["SUMLEV"] == 50]

df = df.set_index("STNAME").groupby(level=0)["CENSUS2010POP"].agg(np.average)

df.head()

STNAME
Alabama        71339.343284
Alaska         24490.724138
Arizona       426134.466667
Arkansas       38878.906667
California    642309.586207
Name: CENSUS2010POP, dtype: float64

The `pd.cut()` function takes a continuous array (or Series) and bins the values into a specified number of intervals (bins). It returns a Series of categorical objects indicating the bin each element falls into.

In [10]:
pd.cut(df, 10)

STNAME
Alabama                   (11706.087, 75333.413]
Alaska                    (11706.087, 75333.413]
Arizona                 (390320.176, 453317.529]
Arkansas                  (11706.087, 75333.413]
California              (579312.234, 642309.586]
Colorado                 (75333.413, 138330.766]
Connecticut             (390320.176, 453317.529]
Delaware                (264325.471, 327322.823]
District of Columbia    (579312.234, 642309.586]
Florida                 (264325.471, 327322.823]
Georgia                   (11706.087, 75333.413]
Hawaii                  (264325.471, 327322.823]
Idaho                     (11706.087, 75333.413]
Illinois                 (75333.413, 138330.766]
Indiana                   (11706.087, 75333.413]
Iowa                      (11706.087, 75333.413]
Kansas                    (11706.087, 75333.413]
Kentucky                  (11706.087, 75333.413]
Louisiana                 (11706.087, 75333.413]
Maine                    (75333.413, 138330.766]
Maryland     